# Implement with WGF

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import optuna
import jax
import jax.numpy as jnp
import optuna
import json
from functools import partial
from jax import jit, vmap, vmap
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio
import numpy as np

In [2]:
import sys
from pathlib import Path
# Add project root to path for imports
ROOT = Path.cwd().parent  
sys.path.append(str(ROOT))

from optimize import objective


In [3]:
%load_ext autoreload
%autoreload 2

## Optimize Variables

In [4]:
df = pd.read_parquet("../data/play_types.parquet")
df = df.drop(index=[29, 84], errors='ignore')
df = df.reset_index(drop=True)
no_fastbreak = df[~df['play_action'].isin(['N/A', 'transition'])]

In [5]:
STUDY_NAME = "nba-defensive-optimization-v3"
STORAGE_NAME = f"sqlite:///{STUDY_NAME}.db"

print("Starting optimization study...")

study = optuna.create_study(
    study_name=STUDY_NAME,
    storage=STORAGE_NAME,
    directions=["minimize", "minimize"],
    load_if_exists=True # Set to False for the first run!
)

# Run 100 trials (adjust this number based on how much time you have)
study.optimize(lambda trial: objective(trial, no_fastbreak[:45]), n_trials=50)

print("\nOptimization Finished!")
print(f"Total trials recorded: {len(study.trials)}")

# Print the best trade-offs (Pareto Front)
print("\nPareto Front (Best Trials):")
for trial in study.best_trials:
    print(f"  Trial {trial.number}:")
    print(f"  Losses (Pressure, Smoothness): {trial.values}")
    
# Try to visualize the Pareto front
try:
    import optuna.visualization as vis
    fig = vis.plot_pareto_front(study)
    fig.show()
    
except ImportError:
    print("Skipping visualization. (Install plotly/kaleido to view the Pareto front graph).")

Starting optimization study...


[I 2026-03-01 23:06:11,468] Using an existing study with name 'nba-defensive-optimization-v3' instead of creating a new one.
[I 2026-03-01 23:06:18,259] Trial 50 finished with values: [1.5675679445266724, 0.2712743878364563] and parameters: {'ist_q_exp': 3.6472860487986845, 'ist_o_exp': 1.0433315985674176, 'ist_weight': 38.64732779838517, 'ist_k_smooth': 15.330639705901339}.
[I 2026-03-01 23:06:21,253] Trial 51 finished with values: [1.3704034090042114, 0.27202266454696655] and parameters: {'ist_q_exp': 3.7825492500468165, 'ist_o_exp': 2.456493292621067, 'ist_weight': 15.106304827227703, 'ist_k_smooth': 8.29838010144404}.
[I 2026-03-01 23:06:23,915] Trial 52 finished with values: [1.3580025434494019, 0.2719400227069855] and parameters: {'ist_q_exp': 3.681931802129557, 'ist_o_exp': 1.8912530302370683, 'ist_weight': 13.189774493789194, 'ist_k_smooth': 12.440595542190866}.
[I 2026-03-01 23:06:26,641] Trial 53 finished with values: [1.3291560411453247, 0.2733653485774994] and parameters: {


Optimization Finished!
Total trials recorded: 100

Pareto Front (Best Trials):
  Trial 26:
  Losses (Pressure, Smoothness): [1.5054281949996948, 0.26864877343177795]
  Trial 36:
  Losses (Pressure, Smoothness): [1.3459278345108032, 0.2710442543029785]
  Trial 37:
  Losses (Pressure, Smoothness): [1.4908117055892944, 0.26934242248535156]
  Trial 40:
  Losses (Pressure, Smoothness): [1.6154989004135132, 0.2682989835739136]
  Trial 53:
  Losses (Pressure, Smoothness): [1.3291560411453247, 0.2733653485774994]
  Trial 55:
  Losses (Pressure, Smoothness): [1.4519565105438232, 0.26986292004585266]
  Trial 60:
  Losses (Pressure, Smoothness): [1.4434244632720947, 0.2705918252468109]
  Trial 84:
  Losses (Pressure, Smoothness): [1.3316248655319214, 0.27291297912597656]
  Trial 94:
  Losses (Pressure, Smoothness): [1.3381810188293457, 0.27201998233795166]
